In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import os

# Multi-layer relative fallback tree schema to point safely to your repository data directory
possible_paths = [
    os.path.join('..', 'data', 'netflix_titles.csv'),
    os.path.join('data', 'netflix_titles.csv'),
    'netflix_titles.csv'
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError("CRITICAL ERROR: Could not locate 'netflix_titles.csv' in any portfolio repository configuration layout.")

print(f"Success! Loading baseline metrics matrix from: {data_path}")
netflix_df = pd.read_csv(data_path)
netflix_df.head()

In [3]:
netflix_df.columns

check for the basic information

In [4]:
netflix_df.info()

there are a lot of null values in the director, cast and country cols

Task 1: The "Ghost Content" Audit (Missing Values)

In [5]:
round(netflix_df.isna().sum()/len(netflix_df),2)

removing the unnecessary cols - show_id and description

In [6]:
netflix_df.drop(['show_id','description'],axis=1,inplace=True)

In [7]:
netflix_df.head()

Task 2: The Content Type Skew (Descriptive Stats)

In [8]:
netflix_df.groupby(by='type').count()

In [9]:
round(netflix_df.groupby(by='type').size()/len(netflix_df),2)

In [10]:
netflix_df['type'].size

In [11]:
len(netflix_df)

70% of netflix catalog is movies

Task 3: The Content Decay Analysis (Dispersion)

checking the dispersion of the releases

In [12]:
rng = netflix_df['release_year'].max() - netflix_df['release_year'].min()
print(f'range of the release year: {rng} years')

In [13]:
netflix_df['release_year'].min()

netflix has a catalog ranging from 96 years. from 1925 to 2021 movies.

In [14]:
var = netflix_df['release_year'].var()
print(var)

In [15]:
std = netflix_df['release_year'].std()
print(std)

In [16]:
plt.hist(netflix_df['release_year'])

though the catalog spans 100 years, most of the movies are from 2010-2020 (recent movies)

Task 1: The Maturity Profile Audit (Audience Breakdown)

In [17]:
netflix_df['rating'].isna().sum()

there are 4 missing values in rating. as the count is very low, will fill these with mode of the rating

In [18]:
mode = netflix_df['rating'].mode()
print(mode)

In [19]:
mode[0]

In [20]:
netflix_df['rating'].fillna(value = mode[0],inplace = True)

In [21]:
netflix_df['rating'].isna().sum()

missing values are handled in rating

In [22]:
netflix_df['rating'].value_counts()

In [23]:
rating = netflix_df['rating'].value_counts(normalize=True)
print(round(rating*100,2))

In [24]:
rating.plot(kind='bar')

majority of the shows or movies are for mature audience (around >17 age group). the catalog mostly focuses on trending concepts that appeals these audience

Task 2: The Multi-Variable Audience Matrix (Type vs. Rating)

In [25]:
audience_matrix = pd.crosstab(netflix_df['type'],netflix_df['rating'],normalize=True)
print(round(audience_matrix*100),2)

netflix is pushing its mature content to its short term format (movies) than tv shows. 


Task 3: Visualizing the Core Pillars

In [26]:
import seaborn as sns

In [27]:
plt.figure(figsize=(15,7))
sns.heatmap(round(audience_matrix*100,2),annot=True,cmap = 'viridis')

netflix has a  share of movies around 70% by volume. 23% out of 70% of movies translates to 33% of TV-MA rating. Where as for Tvshows, 13% out of 30% translates to 43% ot TV-Ma rating. This says netflix is pushing more mature content in tv shows than movies. the catalog caters to various sections of the audiences in the movies than tv shows. most of the tv shows they cater to are TV-Ma and TV14 audiences.

**The Intermediate Playbook (Content Decay & Velocity)**

In [28]:
#converting date into proper datetime format
release_dt = pd.to_datetime(netflix_df['date_added'])
release_dt

In [29]:
netflix_df['date_added'] = release_dt

In [30]:
netflix_df.info()

In [31]:
#creating two cols
netflix_df['year'] = netflix_df['date_added'].dt.year
netflix_df['month'] = netflix_df['date_added'].dt.month

In [32]:
netflix_df.head()

how many titles are added each year?

In [33]:
titles_per_year = netflix_df.groupby('year').count()['type']
titles_per_year

In [34]:
titles_per_year.plot(kind='bar')

though netflix started as streaming service in 2007, they have started adding more titles from 2016 coz it announced its expansion n 130 countries in 2016. 2019 saw the spike where more titles added ever due to lockdown. it helped all of us cope up during lockdown by adding more titles which indirectly relates to more viewership.
Upto 2021, their acquisition of the titles increased by 2021 compared to initial stages. As the lockdown is off, their velocity decreased bit.

**Advanced Strategies**

Genre explosion

In [35]:
netflix_df['listed_in'].value_counts()

since one movie has multiple genres, lets split the genres and map movies against each

In [36]:
netflix_df['listed_in']=netflix_df['listed_in'].str.split(', ')

In [37]:
netflix_df.head()

In [38]:
genre_df = netflix_df.explode('listed_in')

In [39]:
titles_by_genres = genre_df['listed_in'].value_counts()

In [40]:
plt.figure(figsize=(14,6))
title_by_genres.plot(kind='bar')

netflix has around 2500+ international movies catalog compared to other genres. This is part of their strategy to cater and survive their expansion in 130 countries and retain subscribers across various nations. we see very less TV shows and Movies as genre, this might be incorrect tagging or emergence of new tv genres.

In [41]:
netflix_df

**Tier 5: Content Longevity & Binge-Watching Engineering**

In [43]:
movies_df = netflix_df[netflix_df['type']=='Movie']
movies_df.head()

In [70]:
#converting the duration into int for the avg
movies_df['duration'].isna().sum()

In [73]:
movies_df[['dur','dur_unit']]=movies_df['duration'].str.split(' ',n=1,expand=True)

In [74]:
movies_df.head()

In [85]:
#converting dur to numeric
movies_df['dur']=pd.to_numeric(movies_df['dur'],errors='coerce').astype('Int64')

In [86]:
movies_avg_dur = movies_df['dur'].mean()
movies_avg_dur

In [89]:
movies_df['dur'].dtype

In [91]:
movies_df['dur'].fillna(int(movies_avg_dur),inplace=True)

In [94]:
movies_df['dur'].isna().sum()

lets create a duration bucket and distribute the movies across it

In [100]:
plt.hist(movies_df['dur'],bins=[30,60,90,120,150,180,240,270,300])

majority of the movies span between 90 and 120 mins. there are movies that are less than 30 mins long all the way to 3 mins. there are movies that are nearly 240 mins long( 4 hrs). bandersnatch is the interactive movie that takes long time to complete (around 5 hrs).

In [110]:
tv_df = netflix_df[netflix_df['type']=='TV Show']
tv_df.head()

In [114]:
tv_df[['dur','dur_unit']] = tv_df['duration'].str.split(' ',n=1,expand=True)
tv_df.head()

In [116]:
tv_df['dur'].isna().sum()

In [121]:
tv_df['dur']=tv_df['dur'].astype(int)

In [128]:
tv_df['dur'].mode()

In [130]:
mode_seasons = tv_df['dur'].mode()[0]
mode_seasons

mode of no of seasons is 1

In [131]:
plt.hist(tv_df['dur'])

the seasons distribution is right skewed. around 2100+ tv shows has only 1 or 2 seasons. only a few tv shows has 10 and above seasons. the very long duration tv shows (many seasons) is not so profitable. so it focussed more on tv shows with limited seasons as 1 or 2.

In [133]:
tv_df['dur'].value_counts(normalize=True)

67% of the tv shows are short lived - 1 season. if limited season indicates to short lived then it is in aligned.